# Deviation Analysis
## Compare Actual Performance vs Goals

This notebook analyzes how metrics deviate from established goals,
categorizing performance into ranges and identifying problem areas.

In [ ]:
# Parameters (injected by Papermill)
data_path = "data_etl.csv"
output_dir = "./results"
metric = "Actual_SL"
dimensions = '["LOB", "Date"]'
goal_default = 0.80
goal_overrides = '{}'
thresholds = '{"significantly_above": 10, "slightly_above": 5, "within_range": 0, "slightly_below": -5, "significantly_below": -10}'
special_rules = '[]'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os

# Parse JSON parameters
dimensions = json.loads(dimensions) if isinstance(dimensions, str) else dimensions
goal_overrides = json.loads(goal_overrides) if isinstance(goal_overrides, str) else goal_overrides
thresholds = json.loads(thresholds) if isinstance(thresholds, str) else thresholds
special_rules = json.loads(special_rules) if isinstance(special_rules, str) else special_rules

os.makedirs(output_dir, exist_ok=True)

print(f"Data: {data_path}")
print(f"Metric: {metric}")
print(f"Dimensions: {dimensions}")
print(f"Default Goal: {goal_default}")
print(f"Overrides: {goal_overrides}")

In [ ]:
# Load data
df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Validate metric exists
if metric not in df.columns:
    raise ValueError(f"Metric '{metric}' not found. Available: {list(df.columns)}")

# Filter to rows with valid metric data
analysis_df = df[df[metric].notna()].copy()
print(f"Rows with valid {metric}: {len(analysis_df):,}")

In [ ]:
# Apply goals based on dimension overrides
def get_goal(row):
    for dim in dimensions:
        if dim in row.index and str(row[dim]) in goal_overrides:
            return goal_overrides[str(row[dim])]
    return goal_default

analysis_df['Goal'] = analysis_df.apply(get_goal, axis=1)

# Calculate deviation in percentage points
analysis_df['Deviation_Pts'] = (analysis_df[metric] - analysis_df['Goal']) * 100

print("Goal distribution:")
print(analysis_df['Goal'].value_counts())

In [ ]:
# Categorize deviations
def categorize_deviation(row):
    deviation = row['Deviation_Pts']
    actual = row[metric]
    goal = row['Goal']
    
    # Check special rules first
    for rule in special_rules:
        # Simple condition parsing (goal == X AND actual > Y)
        condition = rule.get('condition', '')
        if 'goal ==' in condition and 'actual >' in condition:
            parts = condition.split('AND')
            goal_check = float(parts[0].split('==')[1].strip())
            actual_check = float(parts[1].split('>')[1].strip())
            if goal == goal_check and actual > actual_check:
                return rule.get('override_category', 'Significantly Above')
    
    # Standard categorization
    if deviation >= thresholds['significantly_above']:
        return 'Significantly Above'
    elif deviation >= thresholds['slightly_above']:
        return 'Slightly Above'
    elif deviation >= thresholds['slightly_below']:
        return 'Within Range'
    elif deviation >= thresholds['significantly_below']:
        return 'Slightly Below'
    else:
        return 'Significantly Below'

analysis_df['Category'] = analysis_df.apply(categorize_deviation, axis=1)

# Category order (bottom to top for stacked charts)
category_order = [
    'Significantly Below',
    'Slightly Below',
    'Within Range',
    'Slightly Above',
    'Significantly Above'
]

print("\n=== Overall Distribution ===")
print(analysis_df['Category'].value_counts().reindex(category_order))

In [ ]:
# Create summary by primary dimension
primary_dim = dimensions[0] if dimensions else None

if primary_dim and primary_dim in analysis_df.columns:
    summary = analysis_df.groupby([primary_dim, 'Category']).size().unstack(fill_value=0)
    summary = summary.reindex(columns=category_order, fill_value=0)
    summary['Total'] = summary.sum(axis=1)
    summary['Goal'] = [f"{goal_overrides.get(str(idx), goal_default)*100:.0f}%" for idx in summary.index]
    
    print(f"\n=== Summary by {primary_dim} ===")
    display(summary)

In [ ]:
# Visualization: Stacked bar chart
if primary_dim and primary_dim in analysis_df.columns:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Sort by total descending
    dim_order = summary['Total'].sort_values(ascending=False).index.tolist()
    
    # Colors: red -> orange -> green -> light blue -> blue
    colors = ['#e74c3c', '#f39c12', '#27ae60', '#5dade2', '#2980b9']
    
    # Plot data
    plot_data = summary.loc[dim_order, category_order]
    plot_data.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5, width=0.8)
    
    # Customize
    ax.set_ylabel(f'Number of Records', fontsize=12)
    ax.set_xlabel(primary_dim, fontsize=12)
    ax.set_title(f'{metric} Performance by {primary_dim}\n(Categorized by deviation from goal)', fontsize=14, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    
    # Add totals on top
    for i, dim_val in enumerate(dim_order):
        total = summary.loc[dim_val, 'Total']
        ax.annotate(f'n={total}', xy=(i, total + 5), ha='center', fontsize=8)
    
    ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/deviation_by_{primary_dim.lower()}.png", dpi=150, bbox_inches='tight')
    print(f"Saved: {output_dir}/deviation_by_{primary_dim.lower()}.png")
    plt.show()

In [ ]:
# Save analysis data
analysis_df.to_csv(f"{output_dir}/analysis_output.csv", index=False)

if primary_dim:
    summary.to_csv(f"{output_dir}/summary_by_{primary_dim.lower()}.csv")

print("\n=== Files Saved ===")
print(f"  {output_dir}/analysis_output.csv")
if primary_dim:
    print(f"  {output_dir}/summary_by_{primary_dim.lower()}.csv")
    print(f"  {output_dir}/deviation_by_{primary_dim.lower()}.png")

In [ ]:
# Final summary
print("\n" + "="*60)
print("DEVIATION ANALYSIS COMPLETE")
print("="*60)

total = len(analysis_df)
meeting_goal = len(analysis_df[analysis_df['Deviation_Pts'] >= thresholds['slightly_below']])
sig_below = len(analysis_df[analysis_df['Category'] == 'Significantly Below'])

print(f"\nTotal records analyzed: {total:,}")
print(f"Meeting/Exceeding goal: {meeting_goal:,} ({meeting_goal/total*100:.1f}%)")
print(f"Significantly below goal: {sig_below:,} ({sig_below/total*100:.1f}%)")

print(f"\nDistribution:")
for cat in category_order:
    count = (analysis_df['Category'] == cat).sum()
    pct = count / total * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")